In [ ]:
from dotenv import load_dotenv

from langchain_teddynote import logging
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_teddynote.document_compressors import LLMChainExtractor, LLMChainFilter, EmbeddingsFilter
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_community.document_transformers import EmbeddingsRedundantFilter

from langchain_openai import ChatOpenAI

In [ ]:
load_dotenv()

In [ ]:
logging.langsmith("langchain-10")

In [ ]:
# 문서를 예쁘게 출력하기 위한 도우미 함수
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"문서 {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

In [ ]:
loader = TextLoader("./data/appendix-keywords.txt")

In [ ]:
text_splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=0)
texts = loader.load_and_split(text_splitter)

In [ ]:
embeddings = OpenAIEmbeddings()

In [ ]:
retriever = FAISS.from_documents(texts, embeddings).as_retriever()

In [ ]:
docs = retriever.invoke("Semantic Search 에 대해서 알려줘.")

In [ ]:
pretty_print_docs(docs)

ContextualCompression

In [ ]:
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

In [ ]:
compressor = LLMChainExtractor.from_llm(llm)

In [ ]:
compression_retriever1 = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever,
)

In [ ]:
pretty_print_docs(retriever.invoke("Semantic Search 에 대해서 알려줘."))

print("=========================================================")
print("============== LLMChainExtractor 적용 후 ==================")

compressed_docs1 = (
    compression_retriever1.invoke(  # 컨텍스트 압축 리트리버를 사용하여 관련 문서 검색
        "Semantic Search 에 대해서 알려줘."
    )
)
pretty_print_docs(compressed_docs1)

LLMChainFilter: LLM을 활용한 문서 필터링

In [ ]:
_filter = LLMChainilter.from_llm(llm)

In [ ]:
compression_retriever2 = ContextualCompressionRetriever(
    base_compresspr=_filter, 
    base_retriever=retriever
)

In [ ]:
compressed_docs2 = compression_retriever2.invoke(
    # 쿼리
    "Semantic Search 에 대해서 알려줘."
)

In [ ]:
pretty_print_docs(compressed_docs2)

EmbeddingsFilter

In [ ]:
embeddings = OpenAIEmbeddings()

In [ ]:
embeddings_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.86)

In [ ]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter, 
    base_retriever=retriever
)

In [ ]:
compressed_docs = compression_retriever.invoke("Semantic Search 에 대해서 알려줘.")

In [ ]:
pretty_print_docs(compressed_docs)

파이프라인: Compressor + Document transformer

In [ ]:
splitter = CharacterTextSpliter(chunk_size=300, chunk_overlap=0)

In [ ]:
redundant_filter = EmbeddingsRedundantFilter(embeddings=embeddings)

In [ ]:
relevant_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.86)

In [ ]:
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[
        splitter, 
        redundant_filter, 
        relevant_filter, 
        LLMChainExtractor.from_llm(llm)
    ]
)

In [ ]:
# ContextualCompressionRetriever 초기화

compression_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor, 
    base_retriever=retriever
)

In [ ]:
compressed_docs = compression_retriever.invoke("Semantic Search 에 대해서 알려줘.")

In [ ]:
pretty_print_docs(compressed_docs)